# 07 — Modelado binario `normal_sinus` vs `arrhythmia_or_abnormal`

Iteración binaria de modelado. Reformula la tarea multiclase previa como detección binaria de anormalidad/arritmia.

**Pre-requisitos**

1. `scripts/04_audit_binary_rhythm_dataset.py` ya generó los CSVs descriptivos.
2. `scripts/05_build_binary_rhythm_modeling_dataset.py` ya produjo `data/processed/binary_rhythm_modeling_dataset.parquet` (incluye `rhythm_binary` + features RR rolling).

**Reglas obligatorias**

- Split 80/20 por `case_id` con cobertura de ambas clases en train y test.
- `beat_type`, `case_id`, `rhythm_label`, `rhythm_binary`, `rhythm_classes`, `bad_signal_quality*` NUNCA entran como features.
- CV interna por grupo (`StratifiedGroupKFold` con fallback a `GroupKFold`).
- Test congelado: una sola evaluación al final.
- Selección de umbral SOLO sobre train (vía `cross_val_predict`).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.binary_search import (
    PRIMARY_BINARY_SCORING,
    build_binary_cv_splitter,
    build_binary_model_registry,
    classify_binary_features,
    get_positive_class_score,
    load_binary_modeling_dataset,
    make_binary_group_train_test_split_with_coverage,
    run_binary_search_for_model,
    select_threshold_youden_j,
)
from src.utils import get_logger, set_seed

set_seed(config.RANDOM_SEED)
logger = get_logger("nb07")
sns.set_theme(context="notebook", style="whitegrid")

## 2. Carga y diagnóstico inicial

In [ ]:
df = load_binary_modeling_dataset()
print("shape:", df.shape)
print("cases:", df[config.CASE_ID_COLUMN].nunique())
df[config.BINARY_TARGET_COLUMN].value_counts(normalize=True).round(3)

## 3. Clasificación de columnas y bloqueo de leakage

In [ ]:
cls = classify_binary_features(df)
for k, v in cls.items():
    print(f"{k}: {len(v)}")
print()
print("leakage excluido:", cls["leakage_excluded"])
print("high cardinality:", cls["high_cardinality_excluded"])
print("constantes:", cls["constant_excluded"])
print("too_missing:", cls["too_missing_excluded"])

## 4. Split por `case_id`

In [ ]:
numeric = cls["numeric_features"]
categorical = cls["categorical_features"]

X_df = df[numeric + categorical]
y = df[config.BINARY_TARGET_COLUMN].to_numpy()
groups = df[config.CASE_ID_COLUMN].to_numpy()

train_idx, test_idx, split_info = make_binary_group_train_test_split_with_coverage(
    X_df, y, groups, test_size=0.2, random_state=config.RANDOM_SEED, max_attempts=500,
)
X_train, X_test = X_df.iloc[train_idx], X_df.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train = groups[train_idx]

assert set(groups[train_idx]).isdisjoint(set(groups[test_idx]))
print(json.dumps({k: v for k, v in split_info.items() if k not in ("train_groups", "test_groups")}, indent=2, default=str))
print(f"train cases={len(split_info['train_groups'])}, test cases={len(split_info['test_groups'])}")
print(f"train rows={len(train_idx)}, test rows={len(test_idx)}")

## 5. CV interna por grupo

In [ ]:
N_SPLITS = 5
cv, cv_name, n_splits_eff = build_binary_cv_splitter(groups_train, y_train, n_splits=N_SPLITS)
print("splitter:", cv_name, "| n_splits efectivo:", n_splits_eff)

## 6. Búsqueda multi-modelo

Para uso interactivo usa pocas iteraciones (`N_ITER=5`). Para la corrida formal usar `scripts/06_run_binary_rhythm_model_search.py --n-iter 30 --n-splits 5`.

In [ ]:
registry = build_binary_model_registry()
print("modelos disponibles:", list(registry.keys()))

MODELS_TO_RUN = ["dummy_most_frequent", "logreg_balanced", "hist_gradient_boosting"]
N_ITER = 5

results = []
fitted = {}
for name in MODELS_TO_RUN:
    logger.info("===== %s =====", name)
    spec = registry[name]
    res = run_binary_search_for_model(
        spec=spec,
        X_train=X_train, y_train=y_train, groups_train=groups_train,
        numeric_features=numeric, categorical_features=categorical,
        cv=cv, n_iter=N_ITER,
        random_state=config.RANDOM_SEED, n_jobs=-1,
    )
    fitted[name] = res.best_estimator
    results.append({
        "model": res.model,
        "best_cv_score_primary": res.best_cv_score_primary,
        "fit_seconds": res.fit_seconds,
    })
pd.DataFrame(results).round(3)

## 7. Evaluación en test

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    balanced_accuracy_score, f1_score, roc_auc_score, average_precision_score,
)

for name, est in fitted.items():
    print(f"=== {name} ===")
    y_pred = est.predict(X_test)
    print("balanced_accuracy:", round(balanced_accuracy_score(y_test, y_pred), 3))
    print("f1_abnormal:", round(f1_score(y_test, y_pred, pos_label=config.BINARY_POSITIVE_CLASS, zero_division=0), 3))
    y_score = get_positive_class_score(est, X_test)
    if y_score is not None:
        y_true_bin = (y_test == config.BINARY_POSITIVE_CLASS).astype(int)
        print("ROC-AUC:", round(roc_auc_score(y_true_bin, y_score), 3))
        print("AP:    ", round(average_precision_score(y_true_bin, y_score), 3))
    print(classification_report(y_test, y_pred,
                                labels=[config.BINARY_NEGATIVE_CLASS, config.BINARY_POSITIVE_CLASS],
                                zero_division=0))

## 8. Notas

- Para outputs persistentes (CSVs y figuras), correr el script CLI `scripts/06_run_binary_rhythm_model_search.py`. Este notebook es para inspección rápida.
- El umbral de decisión NO debe seleccionarse sobre `y_test`. El script CLI lo selecciona vía `cross_val_predict` sobre train con la misma CV por grupo.